# LLM 기반 유사 판례 검색 및 법률 쟁점 요약 실험

이 노트북은 기존 Sentence-BERT 유사 판례 검색 프로젝트를 LLM 프로젝트로 확장하기 위한 실험 파일입니다.

중요한 점은 이 프로젝트에서 LLM을 새로 대규모 학습시키는 것이 아니라, 검색된 판례를 근거로 LLM이 쟁점을 요약하도록 만드는 RAG 구조를 사용한다는 것입니다.

- Retrieval: Sentence-BERT 임베딩으로 유사 판례 검색
- Augmentation: 검색된 판례 Top-K를 LLM 프롬프트에 근거 자료로 제공
- Generation: Ollama Gemma2 모델이 사건 요약, 쟁점, 공통점, 차이점을 생성

공개 시연 버전에서는 AI Hub 원본/가공 데이터를 포함하지 않고, 합성 샘플 데이터만 사용합니다.

## 1. 실험 환경 설정

아래 코드는 프로젝트 루트를 기준으로 모듈을 불러옵니다. VS Code에서 실행할 때 커널은 `C:\\Python310\\PROJECT2\\.venv\\Scripts\\python.exe`를 선택하세요.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(r"C:\Python310\PROJECT2")
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import SAMPLE_DATA_PATH
from src.data import load_cases, documents_from_cases
from src.embedding import LegalEmbedder
from src.search import semantic_search
from src.llm import build_legal_analysis_prompt, generate_ollama_analysis

PROJECT_ROOT, SAMPLE_DATA_PATH

## 2. 공개 시연용 합성 판례 데이터 로드

AI Hub 데이터는 이용 조건상 GitHub나 외부 배포 환경에 포함하지 않습니다. 따라서 이 노트북의 공개 실행 예시는 합성 샘플 데이터로 진행합니다.

In [ ]:
cases = load_cases(SAMPLE_DATA_PATH)
print(f"샘플 판례 수: {len(cases):,}건")
display(cases[["id", "case_name", "category", "issues"]].head(10))

In [ ]:
category_counts = cases["category"].value_counts().rename_axis("category").reset_index(name="count")
display(category_counts)
category_counts.plot.bar(x="category", y="count", figsize=(10, 4), title="공개 시연용 합성 판례 분야 분포")

## 3. 판례 문서 임베딩 생성

각 판례의 사건명, 분야, 쟁점, 요약, 본문을 하나의 검색 문서로 합친 뒤 Sentence-BERT 임베딩으로 변환합니다.

이 단계가 LLM 자체 학습은 아니지만, LLM이 사용할 근거 판례를 찾기 위한 검색 모델 단계입니다.

In [ ]:
documents = documents_from_cases(cases)
print(documents[0][:500])

In [ ]:
embedder = LegalEmbedder()
case_embeddings = embedder.encode(documents, batch_size=32)
case_embeddings.shape

## 4. 시연 질의 검색 실험

발표 시연 문장으로 사용할 질의입니다.

> 지나가는 행인을 폭행함

이 질의가 폭행/상해 관련 합성 판례를 상위 결과로 가져오는지 확인합니다.

In [ ]:
query = "지나가는 행인을 폭행함"
results = semantic_search(query, cases, case_embeddings, embedder, top_k=5)
display(results[["case_name", "category", "issues", "similarity", "semantic_similarity", "summary"]])

## 5. LLM 입력 프롬프트 구성

LLM이 판례를 지어내지 않도록 검색 결과 Top-K를 근거 자료로 넣고, 아래 조건을 명시합니다.

- 검색된 판례 정보만 근거로 사용
- 확실하지 않으면 단정하지 않기
- 법률 자문이 아니라 수업 프로젝트용 쟁점 정리로 답변
- 사건 요약, 주요 쟁점, 공통점, 차이점, 추가 확인사항을 구조화해서 출력

In [ ]:
prompt = build_legal_analysis_prompt(query, results)
print(prompt[:2000])

## 6. Ollama Gemma2로 쟁점 요약 생성

실행 전 터미널에서 아래 명령어로 모델을 받아야 합니다.

```powershell
ollama pull gemma2:2b
```

시연 안정성을 위해 `gemma2:2b`를 기본 모델로 사용합니다. `gemma2:9b`는 더 무겁기 때문에 발표 환경에서는 타임아웃이 발생할 수 있습니다.

In [ ]:
RUN_OLLAMA = False  # Ollama가 준비되면 True로 변경
OLLAMA_MODEL = "gemma2:2b"

if RUN_OLLAMA:
    analysis = generate_ollama_analysis(query, results, model=OLLAMA_MODEL)
    print(analysis)
else:
    print("Ollama 실행을 원하면 RUN_OLLAMA를 True로 변경하세요.")

## 7. 프롬프트 튜닝 실험

LLM 프로젝트에서 중요한 실험 포인트는 같은 검색 결과를 주더라도 프롬프트 지시 방식에 따라 출력 품질이 달라진다는 점입니다.

아래는 두 가지 프롬프트 방향입니다.

- 기본형: 쟁점 요약 중심
- 발표형: 사용자가 이해하기 쉬운 설명 중심

In [ ]:
base_prompt = build_legal_analysis_prompt(query, results)

presentation_prompt = base_prompt + """

추가 지시:
- 발표 시연용으로 초보자도 이해할 수 있게 설명해라.
- 어려운 법률 용어는 괄호 안에 짧게 풀어서 설명해라.
- 검색된 판례 중 어떤 판례가 가장 직접적으로 연결되는지 먼저 말해라.
"""

print("기본형 프롬프트 길이:", len(base_prompt))
print("발표형 프롬프트 길이:", len(presentation_prompt))
print(presentation_prompt[-500:])

## 8. 결과 해석

이 프로젝트의 LLM 확장 방식은 다음과 같이 설명할 수 있습니다.

1. 사용자가 사건 내용을 입력한다.
2. Sentence-BERT 임베딩 모델이 의미적으로 유사한 판례를 검색한다.
3. 검색된 판례 Top-K를 LLM 프롬프트에 근거 자료로 넣는다.
4. Gemma2 LLM이 검색 결과를 바탕으로 쟁점, 공통점, 차이점, 추가 확인사항을 요약한다.

따라서 이 시스템은 LLM이 법률 지식을 임의로 생성하는 구조가 아니라, 검색 결과를 근거로 설명을 생성하는 RAG 기반 시스템입니다.